# 15.8 Sequential & session-based recommendation

A user's next action often depends more on the last few actions than on their lifetime average.

This gap-topic notebook adds order to recommender representations. It combines Markov transition counts, recency-weighted embeddings, and query attention while keeping all prediction features strictly before the next item.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1507)


def softmax(scores, temperature=1.0):
    values = np.asarray(scores, dtype=float) / float(temperature)
    values = values - np.max(values)
    weights = np.exp(values)
    return weights / np.sum(weights)


def normalize_rows(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return values / norms


def top_k_indices(scores, k):
    values = np.asarray(scores, dtype=float)
    order = np.argsort(-values)
    return order[:k]


def recall_at_k(scores, positives, k=3):
    values = np.asarray(scores, dtype=float)
    hits = []
    for row, pos in zip(values, positives):
        top = set(top_k_indices(row, k))
        wanted = set(np.atleast_1d(pos).astype(int))
        hits.append(len(top & wanted) / max(1, len(wanted)))
    return float(np.mean(hits))


def dcg_at_k(relevance, k=3):
    rel = np.asarray(relevance, dtype=float)[:k]
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(scores, relevance, k=3):
    values = np.asarray(scores, dtype=float)
    rel = np.asarray(relevance, dtype=float)
    order = np.argsort(-values)[:k]
    ideal = np.argsort(-rel)[:k]
    ideal_dcg = dcg_at_k(rel[ideal], k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(rel[order], k) / ideal_dcg


def mrr_from_scores(scores, target):
    order = np.argsort(-np.asarray(scores, dtype=float))
    rank = int(np.where(order == int(target))[0][0]) + 1
    return 1.0 / rank


def make_f14_ladder(seed=1514):
    rng = np.random.default_rng(seed)
    rungs = []

    item_vectors = np.array([
        [1.0, 0.1, 0.0],
        [0.8, 0.4, 0.0],
        [0.1, 1.0, 0.2],
        [-0.2, 0.3, 0.9],
    ])
    user_vectors = np.array([
        [1.0, 0.2, 0.0],
        [0.2, 1.0, 0.2],
        [-0.1, 0.4, 1.0],
    ])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D1 tiny slate",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.array([0]), np.array([2]), np.array([3])],
        "sessions": [[0, 1, 2], [1, 2, 0], [2, 3, 1]],
        "targets": [2, 0, 1],
        "clicks": np.array([5, 8, 1]),
        "impressions": np.array([100, 100, 20]),
        "eligible": np.array([True, True, True, True]),
        "exposure": np.array([1.0, 0.8, 0.5, 0.3]),
    })

    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    item_vectors = np.column_stack([np.cos(angles), np.sin(angles), 0.35 * np.cos(2.0 * angles)])
    user_angles = np.array([0.1, 1.6, 3.0, 4.7, 5.6])
    user_vectors = np.column_stack([np.cos(user_angles), np.sin(user_angles), 0.25 * np.sin(user_angles)])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D2 synthetic preferences",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 1, 2, 3], [2, 3, 4], [4, 5, 6], [6, 7, 0], [7, 0, 1]],
        "targets": [3, 4, 6, 0, 1],
        "clicks": np.array([12, 20, 18, 6, 4, 3, 9, 7]),
        "impressions": np.array([200, 260, 220, 150, 80, 60, 120, 140]),
        "eligible": np.ones(8, dtype=bool),
        "exposure": np.linspace(1.0, 0.35, 8),
    })

    item_vectors = rng.normal(size=(12, 4))
    item_vectors = normalize_rows(item_vectors)
    user_vectors = rng.normal(size=(7, 4))
    user_vectors = normalize_rows(user_vectors)
    relevance = user_vectors @ item_vectors.T
    relevance = relevance + rng.normal(scale=0.04, size=relevance.shape)
    exposure = rng.uniform(0.15, 1.0, size=12)
    observed = rng.uniform(size=relevance.shape) < exposure[None, :]
    sparse_relevance = np.where(observed, relevance, -0.2)
    rungs.append({
        "name": "D3 sparse noisy logs",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": sparse_relevance,
        "true_relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 2, 5, 7], [1, 2, 4], [3, 8, 9], [10, 2, 0], [6, 6, 11], [4, 5, 6], [9, 1, 3]],
        "targets": [7, 4, 9, 0, 11, 6, 3],
        "clicks": np.array([30, 16, 9, 4, 3, 2, 8, 7, 5, 4, 2, 1]),
        "impressions": np.array([600, 280, 190, 90, 45, 30, 120, 110, 70, 65, 22, 12]),
        "eligible": rng.uniform(size=12) > 0.10,
        "exposure": exposure,
    })

    genres = np.array([
        [1, 0, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1],
        [1, 0, 0, 1, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 1, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
    ], dtype=float)
    item_vectors = normalize_rows(genres + 0.05 * rng.normal(size=genres.shape))
    user_vectors = normalize_rows(np.array([
        [2, 0, 0, 0, 1],
        [1, 2, 0, 0, 0],
        [0, 1, 2, 0, 0],
        [0, 0, 1, 2, 0],
        [0, 0, 0, 1, 2],
        [1, 0, 1, 0, 1],
    ], dtype=float))
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D4 MovieLens-like inline",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 5, 8], [1, 2, 6], [2, 3, 8], [3, 4, 9], [4, 7, 0], [7, 8, 6]],
        "targets": [8, 6, 3, 9, 4, 6],
        "clicks": np.array([50, 34, 25, 18, 17, 14, 9, 7, 6, 5]),
        "impressions": np.array([1000, 700, 520, 430, 320, 250, 150, 110, 90, 80]),
        "eligible": np.ones(10, dtype=bool),
        "exposure": np.array([1.0, 0.9, 0.75, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25, 0.20]),
    })

    head = normalize_rows(rng.normal(loc=0.6, scale=0.25, size=(5, 5)))
    torso = normalize_rows(rng.normal(loc=0.0, scale=0.7, size=(8, 5)))
    cold = normalize_rows(np.array([
        [1, 1, 0, 0, 1],
        [0, 1, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1],
    ], dtype=float))
    item_vectors = np.vstack([head, torso, cold])
    user_vectors = normalize_rows(rng.normal(size=(8, 5)))
    user_vectors[0] = normalize_rows(np.array([[1, 1, 0, 0, 1]], dtype=float))[0]
    relevance = user_vectors @ item_vectors.T
    popularity = np.array([2000, 1500, 1200, 900, 700, 250, 180, 130, 100, 75, 60, 45, 35, 10, 8, 5, 3])
    clicks = np.maximum(1, np.round(popularity * np.clip(0.03 + 0.04 * np.maximum(item_vectors[:, 0], 0.0), 0.01, 0.20))).astype(int)
    exposure = np.clip(popularity / popularity.max(), 0.02, 1.0)
    rungs.append({
        "name": "D5 long-tail cold-start",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 1, 14], [1, 6, 15], [2, 8, 16], [3, 9, 13], [4, 10, 14], [5, 11, 15], [6, 12, 16], [7, 13, 14]],
        "targets": [14, 15, 16, 13, 14, 15, 16, 14],
        "clicks": clicks,
        "impressions": popularity.astype(int),
        "eligible": np.array([True] * 13 + [True, True, False, True]),
        "exposure": exposure,
        "cold_items": np.array([13, 14, 15, 16]),
    })

    return rungs


## The concept, built once: next-item scoring

A recency-weighted profile is

$$h=\sum_t \alpha_t e_{i_t},\qquad \sum_t \alpha_t=1,$$

and a Markov model estimates $P(b\mid a)=C_{ab}/\sum_j C_{aj}$. The reusable method blends Markov probability, recency-vector similarity, and attention similarity.

In [ ]:

def transition_matrix(sessions, n_items, smoothing=0.1):
    counts = np.full((n_items, n_items), float(smoothing))
    for session in sessions:
        for previous, current in zip(session[:-1], session[1:]):
            counts[int(previous), int(current)] += 1.0
    return counts / counts.sum(axis=1, keepdims=True)


def attention_context(query, history_vectors):
    dots = np.asarray(history_vectors, dtype=float) @ np.asarray(query, dtype=float)
    weights = softmax(dots)
    context = weights @ np.asarray(history_vectors, dtype=float)
    return weights, context


def next_item_score(session, item_vectors, all_sessions, query=None, recency_weights=None, markov_weight=0.45, attention_weight=0.30):
    items = np.asarray(item_vectors, dtype=float)
    n_items = items.shape[0]
    if recency_weights is None:
        raw = np.arange(1, len(session) + 1, dtype=float)
        recency_weights = raw / raw.sum()
    history = items[np.asarray(session, dtype=int)]
    h = np.asarray(recency_weights, dtype=float) @ history
    if query is None:
        query = h
    attention_weights, context = attention_context(query, history)
    markov = transition_matrix(all_sessions, n_items)[int(session[-1])]
    recency_scores = items @ h
    attention_scores = items @ context
    combined = markov_weight * markov + (1.0 - markov_weight - attention_weight) * recency_scores + attention_weight * attention_scores
    return {
        "scores": combined,
        "markov": markov,
        "recency": h,
        "attention_weights": attention_weights,
        "context": context,
    }


## Verify the lesson numbers once

For embeddings $(1,0)$, $(0,1)$, $(1,1)$ and weights $(0.2,0.3,0.5)$, the session vector is $h=(0.700,0.800)$. Sessions $[0,1,2]$, $[0,1,1]$, $[1,2,0]$ give $P(2\mid1)=2/3=0.667$ and $P(1\mid1)=0.333$. Attention dots $(1.000,0.200,1.200)$ yield weights $(0.374,0.168,0.457)$ and context $(0.832,0.626)$.

In [ ]:

lesson_items = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
lesson_session = [0, 1, 2]
lesson_weights = np.array([0.2, 0.3, 0.5])
profile = lesson_weights @ lesson_items
assert np.allclose(profile, [0.700, 0.800], atol=0.001)

lesson_sessions = [[0, 1, 2], [0, 1, 1], [1, 2, 0]]
markov = transition_matrix(lesson_sessions, 3, smoothing=0.0)
assert round(float(markov[1, 2]), 3) == 0.667
assert round(float(markov[1, 1]), 3) == 0.333

query = np.array([1.0, 0.2])
weights, context = attention_context(query, lesson_items)
assert [round(float(x), 3) for x in weights] == [0.374, 0.168, 0.457]
assert np.allclose(np.round(context, 3), [0.832, 0.626])
print("profile", np.round(profile, 3))
print("attention", np.round(weights, 3), np.round(context, 3))


## The dataset ladder: D1 to D5

The same F14 recommender ladder is built inline: tiny slate, synthetic preferences, sparse noisy logs, a MovieLens-like genre matrix, and a long-tail cold-start catalog. The single metric here is **MRR**.

In [ ]:

rungs = make_f14_ladder()
for rung in rungs:
    user_shape = rung["user_vectors"].shape
    item_shape = rung["item_vectors"].shape
    rel_shape = rung["relevance"].shape
    sparsity = float(np.mean(rung.get("exposure", np.ones(item_shape[0])) < 0.5))
    print(rung["name"], "users", user_shape, "items", item_shape, "relevance", rel_shape, "low exposure fraction", round(sparsity, 3))
    print("sample relevance row", np.round(rung["relevance"][0, : min(5, item_shape[0])], 3))


## Run the same method across D1-D5

Each session predicts its next item with the same Markov + recency + attention scorer, and MRR rewards how early the held-out next item appears.

In [ ]:

results = []
for rung in rungs:
    scores_by_session = []
    reciprocal_ranks = []
    for session, target in zip(rung["sessions"], rung["targets"]):
        prefix = session[:-1]
        output = next_item_score(prefix, rung["item_vectors"], rung["sessions"])
        scores_by_session.append(output["scores"])
        reciprocal_ranks.append(mrr_from_scores(output["scores"], target))
    metric = float(np.mean(reciprocal_ranks))
    results.append({"name": rung["name"], "metric": metric, "scores": scores_by_session})

for row in results:
    print(f"{row['name']:<28} MRR={row['metric']:.3f}")


## Results visualization

Panels show the first session's next-item ranking per rung, and the curve tracks MRR across the D1-D5 ladder.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, rung, row in zip(flat_axes[:5], rungs, results):
    scores = row["scores"][0]
    order = np.argsort(-scores)[:5]
    colors = ["tomato" if item == rung["targets"][0] else "slategray" for item in order]
    ax.bar(np.arange(len(order)), scores[order], color=colors)
    ax.set_title(rung["name"])
    ax.set_xlabel("candidate rank")
    ax.set_ylabel("next-item score")
flat_axes[5].plot(range(1, 6), [row["metric"] for row in results], marker="o")
flat_axes[5].set_xticks(range(1, 6))
flat_axes[5].set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
flat_axes[5].set_ylim(0.0, 1.05)
flat_axes[5].set_title("MRR curve")
fig.tight_layout()


## Pitfall on D5: shuffled sessions and future leakage

If session order is shuffled, the Markov counts and attention context no longer mean $P(i_t\mid i_{t-1})$. If the target leaks into the prefix, offline MRR becomes unrealistically high. The fix is chronological prefixes with recency and attention over past events only.

In [ ]:

d5 = rungs[-1]
shuffled_sessions = [list(reversed(session)) for session in d5["sessions"]]
wrong_ranks = []
fixed_ranks = []
leaky_ranks = []
for session, target in zip(d5["sessions"], d5["targets"]):
    prefix = session[:-1]
    wrong = next_item_score(prefix, d5["item_vectors"], shuffled_sessions)
    fixed = next_item_score(prefix, d5["item_vectors"], d5["sessions"])
    leaky = next_item_score(session, d5["item_vectors"], d5["sessions"])
    wrong_ranks.append(mrr_from_scores(wrong["scores"], target))
    fixed_ranks.append(mrr_from_scores(fixed["scores"], target))
    leaky_ranks.append(mrr_from_scores(leaky["scores"], target))
print("shuffled-order MRR", round(float(np.mean(wrong_ranks)), 3))
print("chronological MRR", round(float(np.mean(fixed_ranks)), 3))
print("leaky future MRR", round(float(np.mean(leaky_ranks)), 3))


## Evaluate it + practice

- Report **MRR** against a no-skill popularity or random baseline on every rung.
- Sanity check that the D1 worked numbers match the lesson asserts before trusting larger rungs.
- Ablation: replace recency and attention with a uniform lifetime average; the metric should drop on D3-D5.
- Failure signals: head-item collapse, cold items never appearing, or a metric curve that improves only because labels are biased.
- Keep splits chronological or exposure-aware when the lesson's pitfall involves time or logging.

Practice: change the cutoff from 3 to 5 and compare the metric curve.

Practice: add smoothing to rare Markov transitions and compare D5 MRR.